# 02. Pollution & DSC Scoring (Image × Regression Cell)

**Phase 1**: 5종 polluter × level grid × 2 데이터셋 → DSC 측정.

split-first: HF train split을 train/test로 분할 → **train에만** polluter 적용, test는 clean 보존.

ADR-018 polluter 5종: completeness_image / noise_injection / blur (공유) + target_distribution_skew / target_noise (회귀 전용, class_balance·label_swap 대체).

In [ ]:
# ============================================================
# 0-1. Drive 마운트 + GPU 확인
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json
import numpy as np
import pandas as pd
import torch

BASE = '/content/drive/MyDrive/capstone/dsc'
RESULTS_DIR = f'{BASE}/results'
DATA_DIR = f'{BASE}/data/image_regression'
POLLUTED_DIR = f'{BASE}/data/image_regression_polluted'
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

if BASE not in sys.path:
    sys.path.insert(0, BASE)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {device} | torch: {torch.__version__}')

In [ ]:
# ============================================================
# 0-2. 의존성 설치 (Colab)
# ============================================================
%pip install -q datasets timm imagehash opencv-python-headless

In [ ]:
# ============================================================
# 사전등록 메타 (ADR-018) — HuggingFace datasets
# ============================================================
DATASETS = {
    'UTKFace':       {'hf': 'Subh775/UTKFace_demographics_V1', 'target': 'age',          'image_col': 'image', 'role': 'tune'},
    'SCUT_FBP5500':  {'hf': 'MnLgt/scut-fbp5500',              'target': 'beauty_score', 'image_col': 'image', 'role': 'held-out'},
}
TUNE_DS, HELD_DS = 'UTKFace', 'SCUT_FBP5500'
SAMPLE_CAP = 5000   # DSC 계산용 (메모리/시간 절약)
RANDOM_SEED = 42
ML_SPLIT_SEED = 1
ML_TEST_SIZE = 0.2
print(f'데이터셋: {list(DATASETS.keys())} (튜닝={TUNE_DS}, held-out={HELD_DS})')

# Phase 1 level grid (이미지 분류와 동일 5단계)
POLLUTION_LEVELS = [0.1, 0.3, 0.5, 0.7, 0.9]
print('levels:', POLLUTION_LEVELS)

In [ ]:
# ============================================================
# HF 데이터셋 로더 + numpy 변환 (split-first: HF는 train split만 → 자체 분할)
# ============================================================
from datasets import load_dataset

def load_hf_split(ds_name):
    """HF 데이터셋 로드 후 train/test 인덱스 분할 (회귀 — stratify 없음)."""
    meta = DATASETS[ds_name]
    ds = load_dataset(meta['hf'], split='train')
    from sklearn.model_selection import train_test_split
    tr_idx, te_idx = train_test_split(np.arange(len(ds)), test_size=ML_TEST_SIZE,
                                      random_state=ML_SPLIT_SEED)
    return ds, meta, tr_idx, te_idx

def to_arrays(ds, meta, indices, sample_cap=None, random_state=1):
    """주어진 인덱스(예: train) 중 sample_cap개를 (images[np.uint8], targets[float])로."""
    indices = np.asarray(indices)
    if sample_cap and len(indices) > sample_cap:
        rng = np.random.RandomState(random_state)
        indices = indices[rng.permutation(len(indices))[:sample_cap]]
    images, targets = [], []
    for i in indices:
        ex = ds[int(i)]
        img = ex[meta['image_col']]
        if hasattr(img, 'convert'):
            img = img.convert('RGB')
        images.append(np.array(img, dtype=np.uint8))
        targets.append(float(ex[meta['target']]))
    return images, targets, indices

## 1. Polluter 5종 + DSC

In [ ]:
# ============================================================
# 1-1. polluter + DSC import (drive stale 자동 복구)
# ============================================================
import importlib
if not os.path.isdir(f'{BASE}/dsc_framework'):
    from google.colab import drive; drive.mount('/content/drive', force_remount=True)
assert os.path.isdir(f'{BASE}/dsc_framework'), 'Drive 동기화 확인 필요'
if BASE not in sys.path: sys.path.insert(0, BASE)
importlib.invalidate_caches()
for _m in list(sys.modules):
    if _m.startswith('dsc_framework'): del sys.modules[_m]

# pandas 2.x 호환 (dq4ai target polluter가 DataFrame.append 사용)
if not hasattr(pd.DataFrame, 'append'):
    pd.DataFrame.append = lambda s, o, ignore_index=False, **k: pd.concat([s, o], ignore_index=ignore_index)

from dsc_framework import compute_dsc_image_regression
from dsc_framework.image_polluters import (
    CompletenessImagePolluter, NoiseInjectionPolluter, BlurPolluter,
    TargetDistributionSkewImagePolluter, TargetNoiseImagePolluter,
)

def create_polluters(level, seed=RANDOM_SEED):
    return [
        ('completeness_image', CompletenessImagePolluter(level=level, random_seed=seed)),
        ('noise_injection', NoiseInjectionPolluter(level=level, random_seed=seed)),
        ('blur', BlurPolluter(level=level, random_seed=seed)),
        ('target_distribution_skew', TargetDistributionSkewImagePolluter(level=level, random_seed=seed)),
        ('target_noise', TargetNoiseImagePolluter(level=level, random_seed=seed)),
    ]
print('Polluter 5종 정의 완료 (회귀)')

In [ ]:
# ============================================================
# 1-2. split → train 폴루션 → DSC
# ============================================================
from time import time
dsc_rows = []
total_start = time()

for ds_name in DATASETS:
    print(f'\n=== {ds_name} ===')
    ds, meta, tr_idx, te_idx = load_hf_split(ds_name)
    images_clean, targets_clean, _ = to_arrays(ds, meta, tr_idx, sample_cap=SAMPLE_CAP, random_state=1)
    print(f'  train sample {len(images_clean)} (split train={len(tr_idx)}, test={len(te_idx)})')

    res_base = compute_dsc_image_regression(images_clean, targets_clean, sample_cap=SAMPLE_CAP)
    res_base.pop('metrics', None)
    print(f'  baseline DSC = {res_base["score"]} ({res_base["grade"]})')
    dsc_rows.append({'dataset': ds_name, 'polluter': 'none', 'level': 0.0, **res_base})

    for level in POLLUTION_LEVELS:
        for pname, pol in create_polluters(level):
            t0 = time()
            try:
                pi, pt = pol.pollute(images_clean, targets_clean)
                res_p = compute_dsc_image_regression(pi, pt, sample_cap=SAMPLE_CAP)
                res_p.pop('metrics', None)
                pol_dir = f'{POLLUTED_DIR}/{ds_name}/{pname}_{int(level*100)}'
                os.makedirs(pol_dir, exist_ok=True)
                arrs = [np.asarray(im, dtype=np.uint8) for im in pi]
                shapes = {a.shape for a in arrs}
                images_arr = np.stack(arrs) if len(shapes) == 1 else np.array(arrs, dtype=object)
                np.savez_compressed(f'{pol_dir}/data.npz', images=images_arr, targets=np.array(pt, dtype=float))
                dsc_rows.append({'dataset': ds_name, 'polluter': pname, 'level': level, **res_p})
                print(f'  {pname:<26s} L={level:.2f}  DSC={res_p["score"]:6.2f}  Δ={res_p["score"]-res_base["score"]:+.2f}  ({time()-t0:.0f}s)')
            except Exception as e:
                print(f'  {pname:<26s} L={level:.2f}  ERROR: {e}')

print(f'\n총 {len(dsc_rows)}건 ({time()-total_start:.0f}s)')

In [ ]:
# ============================================================
# 2-1. DSC 저장
# ============================================================
df_dsc = pd.DataFrame(dsc_rows)
out = f'{RESULTS_DIR}/dsc_scores_image_regression.csv'
df_dsc.to_csv(out, index=False)
print(f'DSC 저장: {out} (총 {len(df_dsc)}건)')
print('--- 노트북 02 image regression 완료 → 03 실행 ---')
df_dsc.head(12)